# Similarity Search Benchmark: Algorithm Comparison

This notebook benchmarks several algorithms for similarity search on immune receptor sequences at medium dataset sizes (up to 100,000 sequences). Given a dataset of Adaptive Immune Receptor sequences and a Levenshtein distance threshold, each algorithm identifies all pairs of sequences with a similarity below the threshold value. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

For a large-scale benchmark (up to 30 million sequences) comparing implementations of the symmetric deletion algorithm specifically, see the companion notebook `02D_symdel_large_scale.ipynb`.

The notebook is divided into 3 steps as follows:
1. __Configuration:__ select the number of experiment repeats.
2. __Benchmark Setup:__ install dependencies and prepare input data.
3. __Algorithm Benchmark:__ perform benchmark on 6 algorithms (exhaustive search, bk-tree, kd-tree, combinatorial lookup, symmetric deletion, symscan) at threshold `d=1,2`, and across distances `d=1..5` at a fixed size.

Warning: this notebook can take up to 15 minutes to run.


## 1. Configuration

In [ ]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 10 # @param ["1", "10", "30"] {type:"raw"}
high_ram = True # @param {type:"boolean"}

## 2. Benchmark Setup (run time < 5 min)

install dependency

In [ ]:
! pip install -q pyrepseq pybktree symscan

In [ ]:
import os.path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import random
import pybktree
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

import sys

import benchutils
from benchutils import BenchmarkTimeout, run_binary, time_limit, describe_env

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

In [ ]:
benchutils.timeout_seconds = 100

In [ ]:
describe_env()

prepare input data

In [ ]:
N_FILES=6

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'{repo_path}/data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

In [ ]:
# garbage collect following large data import
# freeze imported data to avoid reconsideration during benchmarking
import gc
gc.collect()
gc.freeze()

## 3. Algorithm Benchmark (run time ~ 15 min at n_repeat=1)

exhaustive search implementation

In [ ]:
def exhaustive_search(seqs, max_edits):
  ans = []
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=max_edits)
        if dist <= max_edits:
            ans += [(i, j, dist)]
  return ans

bktree implementation

In [ ]:
def build_index(seqs):
    ans = {}
    for index, seq in enumerate(seqs):
        if seq not in ans:
            ans[seq] = []
        ans[seq].append(index)
    return ans


def bktree(seqs, max_edits=1):
    ans = []
    index = build_index(seqs)

    tree = pybktree.BKTree(levenshtein_distance, np.unique(seqs))
    for x_index, x_seq in enumerate(seqs):
        for edit_distance, y_seq in tree.find(x_seq, max_edits):
            for y_index in index[y_seq]:
                if x_index != y_index:
                    ans.append((x_index, y_index, edit_distance))
    return ans

benchmarking code

In [ ]:
# 100 is the warm up
sizes = [100, 1_000, 3_000, 10_000, 30_000, 100_000]
algorithms = {
    'kdtree':pyrepseq.kdtree,
    'symdel':pyrepseq.symdel,
    'combinatorial_lookup': pyrepseq.hash_based,
    'exhaustive_search': exhaustive_search,
    'bktree': bktree,
    'symscan': symscan.get_neighbors_within,
    }
limits = {}
#    'exhaustive_search_1':100_000,
#    'exhaustive_search_2':100_000,
#    'combinatorial_lookup_2':10_000,
#    'kdtree_2':100_000,
#    'exhaustive_search_2':100_000,
#    'bktree_2': 100_000}
limits.update({f"combinatorial_lookup_{dist}": 0 for dist in [4,5]})

alg_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

def run_alg_exp(distances, sizes=sizes, verbose=False):
    for i in range(n_repeat):
        for size in sizes:
            subset = random.Random(i).sample(data, size)
            for distance in distances:
                for alg_name in algorithms:
                    limit = limits.get(f"{alg_name}_{distance}")
                    if limit is not None and limit < size:
                        if verbose:
                            print(f'Skipping {alg_name} with distance {distance} and size {size:,} because limit is {limit:,}')
                        continue

                    if verbose:
                        print(f'Running {alg_name} with distance {distance} and size {size:,}')

                    # perform
                    start = time.time()
                    try:
                        with time_limit():
                            algorithms[alg_name](subset, distance)
                    except BenchmarkTimeout:
                        print(f'Timeout: {alg_name} with distance {distance} and size {size:,} exceeded {benchutils.timeout_seconds}s')
                        limits[f"{alg_name}_{distance}"] = size - 1
                        continue
                    end = time.time()

                    # record
                    print(f'{size:,}', alg_name, distance, i, round((end-start)*100)/100)
                    alg_result['runtime'].append(end-start)
                    alg_result['algorithm'].append(alg_name)
                    alg_result['input_size'].append(size)
                    alg_result['distance'].append(distance)

In [ ]:
run_alg_exp(distances=[1])

In [ ]:
run_alg_exp(distances=[2])

In [ ]:
pd.DataFrame(alg_result).to_csv('../data/cpu_benchmark.csv')
if colab:
    files.download('cpu_benchmark.csv')

In [ ]:
alg_result = {'runtime':[],'algorithm':[],'input_size':[],'distance':[]}

run_alg_exp(distances=range(1,6), sizes=[10_000], verbose=False)

In [ ]:
pd.DataFrame(alg_result).to_csv('../data/cpu_dist_benchmark.csv')
if colab:
    files.download('cpu_dist_benchmark.csv')